[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/14_Fitting_Least_Squares.ipynb)

# Fitting I: Least Squares and χ²

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 11

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

Measurements are never exact, and physics theories usually come with unknown parameters: Planck's constant, a lifetime, the mass of a planet. **Fitting** is how you combine a model with noisy data to get the best values of those parameters *and* their uncertainties, and to test whether the model describes the data at all.

This week you will build the method of least squares from scratch, then use the SciPy tools that physicists use every day, and apply them to real data: Millikan's photoelectric measurement of $h$, a radioactive decay, global temperatures, and a planet orbiting another star.

## Learning goals

By the end of this notebook you should be able to:

- explain what an uncertainty (error bar) means and what $\chi^2$ measures;
- derive and code the least-squares straight-line fit, and use `np.polyfit`;
- fit any model with `scipy.optimize.curve_fit`, including `sigma` and `absolute_sigma=True`;
- read a covariance matrix: parameter uncertainties and correlations;
- judge a fit with $\chi^2$, degrees of freedom, reduced $\chi^2$, the $p$-value, and residual/pull plots;
- recognize bad fits (wrong model, wrong error bars) and choose initial guesses for nonlinear fits;
- load and explore tabular data with pandas;
- fit a histogram, and see why bins with few or zero counts cause trouble.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import constants as const

rng = np.random.default_rng(8)

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

## 1. Measurements, uncertainties, and the Gaussian

If you measure the same quantity many times, the results scatter. A common model for this scatter is the **Gaussian** (normal) distribution with mean $\mu$ (the true value) and **standard deviation** $\sigma$ (the typical size of the error):

$$p(y) = \frac{1}{\sqrt{2\pi}\,\sigma} \exp\!\left[-\frac{(y-\mu)^2}{2\sigma^2}\right].$$

When you see "$y = 4.3 \pm 0.2$", the $0.2$ is usually such a $\sigma$, and it is drawn as an **error bar**. For a Gaussian, about 68% of measurements fall within $\pm 1\sigma$ of the true value, 95% within $\pm 2\sigma$, and 99.7% within $\pm 3\sigma$. So roughly one in three error bars *should* miss the true curve. Error bars that all touch the curve are a sign that the errors are overestimated.

Let's check the 68% rule by simulation.

In [ ]:
true_value, sigma = 4.3, 0.2
y_meas = rng.normal(true_value, sigma, 100_000)     # 100,000 simulated measurements
within_1 = np.mean(np.abs(y_meas - true_value) < sigma)
within_2 = np.mean(np.abs(y_meas - true_value) < 2 * sigma)
print(f"fraction within 1 sigma: {within_1:.3f};  within 2 sigma: {within_2:.3f}")

### A simulated experiment

Throughout the first part of this notebook we use **simulated** data so that you know the right answer. Imagine dropping a ball and measuring its speed with a sensor at several times. The model is a straight line,

$$v(t) = v_0 + a\,t,$$

with true values $v_0 = 1.5$ m/s and $a = g = 9.81\ \text{m/s}^2$. Each speed measurement has its own uncertainty $\sigma_i$ (the sensor gets worse at higher speed, say), and the simulation adds Gaussian noise of that size.

In [ ]:
t = np.linspace(0.05, 1.0, 12)                    # s
sig_v = 0.15 + 0.25 * t                           # m/s, uncertainty of each point
v_true = 1.5 + 9.81 * t
v = v_true + rng.normal(0, sig_v)                 # simulated measurements

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(t, v, yerr=sig_v, fmt="o", capsize=3, label="simulated data")
ax.plot(t, v_true, "k--", label="true line")
ax.set_xlabel("time $t$ (s)")
ax.set_ylabel("speed $v$ (m/s)")
ax.legend()
plt.show()

## 2. The least-squares idea and $\chi^2$

Which straight line "fits best"? For each data point $(x_i, y_i \pm \sigma_i)$ and a model $f(x; \text{parameters})$, the **residual** $y_i - f(x_i)$ is how far the point lies from the curve. Dividing by $\sigma_i$ measures that distance in units of the error bar. The **chi-squared** adds up the squares:

$$\chi^2 = \sum_{i=1}^{N} \left(\frac{y_i - f(x_i)}{\sigma_i}\right)^2 .$$

The **least-squares** principle says: choose the parameters that make $\chi^2$ as small as possible. Points with small error bars get more say than points with large ones, which is exactly what you want. (Next week you will see that, for Gaussian errors, this is the same as choosing the parameters that make the observed data most probable.)

Let's compute $\chi^2$ for many trial slopes $a$, keeping $v_0$ fixed at its true value, and see where it is smallest.

In [ ]:
def chi2_line(v0, a):
    return np.sum(((v - (v0 + a * t)) / sig_v)**2)

a_trial = np.linspace(8, 11.5, 400)
chi2_scan = np.array([chi2_line(1.5, a) for a in a_trial])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(a_trial, chi2_scan)
ax.set_xlabel(r"trial acceleration $a$ (m/s$^2$)")
ax.set_ylabel(r"$\chi^2$")
plt.show()
print(f"chi2 is smallest at a = {a_trial[np.argmin(chi2_scan)]:.3f} m/s^2")

$\chi^2$ is a parabola in the parameter. The minimum is the best fit, and (as you will show in Your turn 1) the *width* of the parabola tells you the uncertainty: the parameter values where $\chi^2$ rises by 1 above its minimum lie $\pm 1\sigma$ from the best fit. A narrow parabola means a well-determined parameter.

## 3. The straight-line fit, analytically

For $f(x) = b_0 + b_1 x$ you can find the minimum exactly. Set both partial derivatives of $\chi^2$ to zero, using weights $w_i = 1/\sigma_i^2$:

$$\frac{\partial\chi^2}{\partial b_0} = -2\sum_i w_i (y_i - b_0 - b_1 x_i) = 0, \qquad
\frac{\partial\chi^2}{\partial b_1} = -2\sum_i w_i x_i (y_i - b_0 - b_1 x_i) = 0 .$$

With the shorthand sums $S = \sum w_i$, $S_x = \sum w_i x_i$, $S_y = \sum w_i y_i$, $S_{xx} = \sum w_i x_i^2$, $S_{xy} = \sum w_i x_i y_i$, these are two linear equations (the **normal equations**):

$$S\, b_0 + S_x\, b_1 = S_y, \qquad S_x\, b_0 + S_{xx}\, b_1 = S_{xy}.$$

Solving them (with $\Delta = S S_{xx} - S_x^2$):

$$b_0 = \frac{S_{xx}S_y - S_x S_{xy}}{\Delta}, \qquad b_1 = \frac{S\,S_{xy} - S_x S_y}{\Delta}.$$

Propagating the data uncertainties through these formulas gives the parameter uncertainties

$$\sigma_{b_0} = \sqrt{S_{xx}/\Delta}, \qquad \sigma_{b_1} = \sqrt{S/\Delta}.$$

In [ ]:
def line_fit(x, y, sigma):
    """Weighted least-squares straight line y = b0 + b1*x.
    Returns b0, b1, sigma_b0, sigma_b1."""
    w = 1 / sigma**2
    S, Sx, Sy = w.sum(), (w * x).sum(), (w * y).sum()
    Sxx, Sxy = (w * x**2).sum(), (w * x * y).sum()
    Delta = S * Sxx - Sx**2
    b0 = (Sxx * Sy - Sx * Sxy) / Delta
    b1 = (S * Sxy - Sx * Sy) / Delta
    return b0, b1, np.sqrt(Sxx / Delta), np.sqrt(S / Delta)

b0, b1, s0, s1 = line_fit(t, v, sig_v)
print(f"v0 = {b0:.3f} +/- {s0:.3f} m/s      (true 1.5)")
print(f"a  = {b1:.3f} +/- {s1:.3f} m/s^2    (true 9.81)")

### The same thing with `np.polyfit`

`np.polyfit(x, y, deg)` fits a polynomial of degree `deg` by least squares and returns the coefficients **highest power first**, so for a line you get `[slope, intercept]`.

Two pitfalls when you have error bars:

- The weight argument is `w = 1/sigma`, **not** `1/sigma**2` (NumPy squares it internally).
- To get uncertainties, pass `cov="unscaled"`. This returns the **covariance matrix** (explained in Section 5) computed from your error bars as given. Plain `cov=True` instead rescales the uncertainties using the scatter of the points, which is what you want only when you do *not* trust (or do not have) error bars.

In [ ]:
coef, cov = np.polyfit(t, v, 1, w=1 / sig_v, cov="unscaled")
print("polyfit [slope, intercept]:", coef)
print("uncertainties:             ", np.sqrt(np.diag(cov)))

Identical to our hand-written formulas.

### Your turn 1

Using `chi2_line` (with $v_0$ fixed at the best-fit `b0`), scan the slope finely around its minimum and find the two values of $a$ where $\chi^2 = \chi^2_\text{min} + 1$. Is half the distance between them equal to `s1`? (With $v_0$ held fixed you should find it noticeably *smaller* than `s1`. Section 5 explains why: the two parameters are correlated, and `s1` allows for $v_0$ adjusting as $a$ changes. Repeat the scan, but at each trial $a$ use the $v_0$ that minimizes $\chi^2$; now the width should match `s1`.)

In [ ]:
# Your code here

## 4. General fits with `scipy.optimize.curve_fit`

For anything beyond a polynomial you need a general tool. `curve_fit` minimizes $\chi^2$ numerically for **any** model function you write. The model's first argument must be the independent variable, followed by the parameters:

```python
popt, pcov = curve_fit(f, x, y, p0=initial_guess, sigma=errors, absolute_sigma=True)
```

- `popt`: best-fit parameter values (an array).
- `pcov`: their covariance matrix.
- `p0`: starting guess (defaults to all ones; for nonlinear models, give a sensible one, see Section 8).
- `sigma`: the error bar on each $y_i$.
- `absolute_sigma=True`: treat `sigma` as real uncertainties. **Always set this when your error bars are real.** The default (`False`) rescales the uncertainties to make reduced $\chi^2 = 1$ (Section 6), which hides bad error bars.

In [ ]:
def line(x, v0, a):
    return v0 + a * x

popt, pcov = curve_fit(line, t, v, sigma=sig_v, absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))
for name, val, err in zip(["v0", "a"], popt, perr):
    print(f"{name} = {val:.3f} +/- {err:.3f}")

## 5. The covariance matrix: uncertainties and correlations

A **matrix** is a rectangular table of numbers; in NumPy it is just a 2D array. `pcov` is a square matrix with one row and one column per parameter:

$$C = \begin{pmatrix} \sigma_{v_0}^2 & \text{cov}(v_0, a) \\ \text{cov}(a, v_0) & \sigma_a^2 \end{pmatrix}.$$

- The **diagonal** entries $C_{ii}$ are the **variances** (squared uncertainties) of the parameters. That is why we take `np.sqrt(np.diag(pcov))`: `np.diag` pulls the diagonal out into a 1D array.
- The **off-diagonal** entries are **covariances**: they say whether the errors on two parameters are linked. The matrix is symmetric, $C_{ij} = C_{ji}$.

Covariances are easier to read after dividing by the uncertainties, which gives the **correlation coefficient**

$$\rho_{ij} = \frac{C_{ij}}{\sigma_i \sigma_j}, \qquad -1 \le \rho_{ij} \le 1 .$$

$\rho = 0$ means independent errors; $\rho$ near $\pm 1$ means the two parameters can trade off against each other.

In [ ]:
print("covariance matrix:\n", pcov)
corr = pcov / np.outer(perr, perr)       # np.outer(a, b)[i, j] = a[i] * b[j]
print("\ncorrelation matrix:\n", np.round(corr, 3))

The intercept and slope are strongly **anti-correlated**. Picture it: all data are at $t > 0$, so if you tilt the line a little steeper, you must lower the intercept to keep the line passing through the data. A simulation makes this concrete: repeat the whole experiment many times, fit each, and plot the fitted $(v_0, a)$ pairs.

In [ ]:
fits = []
for _ in range(2000):
    v_sim = v_true + rng.normal(0, sig_v)            # a new simulated experiment
    p, _ = curve_fit(line, t, v_sim, sigma=sig_v, absolute_sigma=True)
    fits.append(p)
fits = np.array(fits)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fits[:, 0], fits[:, 1], ".", ms=2, alpha=0.4)
ax.set_xlabel(r"fitted $v_0$ (m/s)")
ax.set_ylabel(r"fitted $a$ (m/s$^2$)")
plt.show()
print("spread of fitted values:   ", fits.std(axis=0))
print("uncertainties from pcov:   ", perr)
print("correlation of fitted values:", np.corrcoef(fits.T)[0, 1].round(3))

The cloud is a tilted ellipse, and its spread and tilt match `pcov`. This is what an uncertainty *means*: if you repeated the experiment, your fitted values would scatter by about that much.

**Tip:** correlations can be reduced by measuring $x$ from the middle of the data. If you fit $v = c_0 + a(t - \bar t)$, then $c_0$ is the speed at the average time and is nearly uncorrelated with $a$. You will see this in the temperature fit later.

## 6. How good is the fit? $\chi^2$, degrees of freedom, $p$-value

If the model is right and the error bars are right, each term $\left(\frac{y_i - f(x_i)}{\sigma_i}\right)^2$ is on average about 1. But fitting $n_\text{par}$ parameters lets the curve bend toward the data, so the expected minimum value is

$$\langle \chi^2_\text{min} \rangle = \nu = N - n_\text{par},$$

where $\nu$ is the number of **degrees of freedom** (ndf). The **reduced chi-squared** $\chi^2/\nu$ should therefore be close to 1:

- $\chi^2/\nu \gg 1$: the model is wrong, or the error bars are too small.
- $\chi^2/\nu \ll 1$: the error bars are too big (or the data were "massaged").
- How close to 1 is close enough depends on $\nu$: for few points, $\chi^2/\nu$ fluctuates a lot.

To make this precise, use the **$\chi^2$ distribution**: the probability distribution of $\chi^2_\text{min}$ over many repeated experiments when everything is correct. The **$p$-value** is the probability of getting a $\chi^2$ at least as large as the one you observed, *if the model and errors are correct*. A tiny $p$ (say below 0.01) suggests something is wrong. `scipy.stats.chi2.sf(x, ndf)` computes it ("sf" = survival function = $1 - $ CDF, the area to the right of $x$).

In [ ]:
from scipy.stats import chi2

def fit_quality(f, x, y, sigma, popt):
    """Print chi2, ndf, chi2/ndf and p-value of a fit."""
    chisq = np.sum(((y - f(x, *popt)) / sigma)**2)
    ndf = len(x) - len(popt)
    p = chi2.sf(chisq, ndf)
    print(f"chi2 = {chisq:.2f}, ndf = {ndf}, chi2/ndf = {chisq/ndf:.2f}, p-value = {p:.3f}")
    return chisq, ndf

chisq, ndf = fit_quality(line, t, v, sig_v, popt)

Let's check the claim that $\chi^2_\text{min}$ follows the $\chi^2$ distribution, using the 2000 simulated experiments from Section 5.

In [ ]:
chi2_values = []
for _ in range(2000):
    v_sim = v_true + rng.normal(0, sig_v)
    p, _ = curve_fit(line, t, v_sim, sigma=sig_v, absolute_sigma=True)
    chi2_values.append(np.sum(((v_sim - line(t, *p)) / sig_v)**2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(chi2_values, bins=50, density=True, alpha=0.6, label="2000 simulated experiments")
xx = np.linspace(0, 35, 300)
ax.plot(xx, chi2.pdf(xx, ndf), "k-", label=rf"$\chi^2$ distribution, $\nu = {ndf}$")
ax.axvline(chisq, color="C3", label="our data")
ax.set_xlabel(r"$\chi^2_\mathrm{min}$")
ax.set_ylabel("probability density")
ax.legend()
plt.show()
print(f"mean of simulated chi2: {np.mean(chi2_values):.2f}  (expected ndf = {ndf})")

## 7. Residual and pull plots

A single number cannot tell you *where* a fit goes wrong. Always look at the **residuals** $y_i - f(x_i)$, or better the **pulls**

$$\text{pull}_i = \frac{y_i - f(x_i)}{\sigma_i},$$

which should scatter randomly around 0 with typical size 1, with no pattern. A standard figure has the data and fit on top and the pulls in a smaller panel underneath, sharing the $x$ axis.

In [ ]:
def plot_fit_with_pulls(x, y, sigma, f, popt, xlabel, ylabel):
    """Data + fitted curve on top, pulls below."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1]})
    ax1.errorbar(x, y, yerr=sigma, fmt="o", ms=4, capsize=2, label="data")
    xx = np.linspace(x.min(), x.max(), 300)
    ax1.plot(xx, f(xx, *popt), "C1-", label="fit")
    ax1.set_ylabel(ylabel)
    ax1.legend()
    pulls = (y - f(x, *popt)) / sigma
    ax2.axhline(0, color="k", lw=0.8)
    ax2.plot(x, pulls, "o", ms=4)
    ax2.set_ylabel("pull")
    ax2.set_xlabel(xlabel)
    plt.show()
    return pulls

pulls = plot_fit_with_pulls(t, v, sig_v, line, popt, "time $t$ (s)", "speed $v$ (m/s)")

`gridspec_kw={"height_ratios": [3, 1]}` makes the top panel three times taller than the bottom one, and `sharex=True` lines up the two $x$ axes.

## 8. When fits go wrong

### Wrong model

*Physics recap.* A ball falling through air feels a drag force growing like $v^2$, so its speed does not grow linearly forever; it approaches a **terminal speed** $v_t$. Starting from rest, the exact solution is $v(t) = v_t \tanh(g t / v_t)$. Let's simulate a light ball ($v_t = 8$ m/s) and fit a straight line anyway.

In [ ]:
g = const.g        # standard gravity, 9.80665 m/s^2
v_t = 8.0          # terminal speed, m/s
t2 = np.linspace(0.05, 1.5, 20)
sig2 = np.full(t2.size, 0.15)
v_drag = v_t * np.tanh(g * t2 / v_t) + rng.normal(0, sig2)

p_lin, _ = curve_fit(line, t2, v_drag, sigma=sig2, absolute_sigma=True)
fit_quality(line, t2, v_drag, sig2, p_lin)
pulls = plot_fit_with_pulls(t2, v_drag, sig2, line, p_lin, "time $t$ (s)", "speed $v$ (m/s)")

Two warning signs: a huge $\chi^2/\nu$ with $p \approx 0$, and pulls that form a **smooth pattern** (a frown) instead of random scatter. A pattern in the pulls tells you the *shape* of the model is wrong. The fix is a better model, here $v_t\tanh(gt/v_t)$ with $v_t$ as a free parameter.

### Wrong error bars

Now the model is right, but the experimenter claims error bars half as big as the real ones (a common mistake), or twice as big.

In [ ]:
v_line = 1.5 + 9.81 * t2 + rng.normal(0, sig2)     # true errors are sig2
for factor in [0.5, 1.0, 2.0]:
    claimed = factor * sig2
    p, _ = curve_fit(line, t2, v_line, sigma=claimed, absolute_sigma=True)
    print(f"claimed sigma = {factor} x true:  ", end="")
    fit_quality(line, t2, v_line, claimed, p)

The best-fit values are the same in all three cases (scaling all $\sigma_i$ by the same factor does not move the minimum), but $\chi^2/\nu$ comes out about 4, 1, and 1/4, and the quoted parameter uncertainties scale with the claimed errors. Here the pulls look random, just too large or too small.

**If you have no error bars at all**, you can *assume* the model is right and estimate a common $\sigma$ from the scatter of the residuals:

$$\hat\sigma^2 = \frac{1}{N - n_\text{par}}\sum_i \big(y_i - f(x_i)\big)^2 .$$

That is what `curve_fit` does when you leave out `sigma` (or set `absolute_sigma=False`), and what `np.polyfit(..., cov=True)` does. The price: you can no longer use $\chi^2$ to test the model, because you forced $\chi^2/\nu = 1$.

### Nonlinear models need initial guesses

For a straight line the minimum of $\chi^2$ is unique and easy to find. For a nonlinear model, `curve_fit` starts at `p0` and walks downhill, and it can get stuck or wander off. Example: a Gaussian peak in a spectrum (simulated), $f(x) = A\exp[-(x-\mu)^2/2s^2]$, with the default `p0` of all ones.

In [ ]:
def gauss(x, A, mu, s):
    return A * np.exp(-(x - mu)**2 / (2 * s**2))

x_spec = np.linspace(0, 100, 101)                 # channel number
sig_spec = np.full(x_spec.size, 0.5)
y_spec = gauss(x_spec, 10, 62, 4) + rng.normal(0, sig_spec)

p_bad, _ = curve_fit(gauss, x_spec, y_spec, sigma=sig_spec, absolute_sigma=True)
print("default p0 (all ones):", np.round(p_bad, 3))

p0 = [y_spec.max(), x_spec[np.argmax(y_spec)], 5.0]     # read guesses off the data
p_good, _ = curve_fit(gauss, x_spec, y_spec, p0=p0, sigma=sig_spec, absolute_sigma=True)
print("sensible p0:          ", np.round(p_good, 3), "  (true: 10, 62, 4)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_spec, y_spec, ".", color="0.5", label="simulated spectrum")
ax.plot(x_spec, gauss(x_spec, *p_bad), label="fit from default p0")
ax.plot(x_spec, gauss(x_spec, *p_good), label="fit from sensible p0")
ax.set_xlabel("channel")
ax.set_ylabel("counts (arb. units)")
ax.legend()
plt.show()

With the default guess, the starting curve sits near $x = 1$, far from the peak at 62, where $\chi^2$ hardly changes as the parameters change. The fit "converged" to nonsense **without any error message**. Always plot your fit!

Good habits for `p0`: read the peak height and position off the data (`max`, `argmax`), estimate widths and periods by eye from a plot, and use physics (known lifetimes, masses). If a fit fails with `RuntimeError: Optimal parameters not found`, the guess is usually the problem; catch it with `try/except` and inspect.

## 9. Tabular data with pandas

Real data often come as tables with named columns. **pandas** is Python's standard library for tables. Its main object is the **DataFrame**: a 2D table whose columns have names and can hold different types (numbers, text, dates).

As an example, load NASA's GISTEMP record of global mean surface temperature. Each row is a year; the columns `Jan` … `Dec` are monthly **temperature anomalies** in °C, meaning differences from the 1951–1980 average. `J-D` is the January–December annual mean. The first line of the file is a title, so skip it with `skiprows=1`, and missing values are written `***`, so tell pandas with `na_values="***"` to turn them into `NaN` ("not a number").

In [ ]:
import pandas as pd

df = pd.read_csv(data_path("GLB.Ts+dSST.csv"), skiprows=1, na_values="***")
df.head()     # first 5 rows

A few essential operations:

- `df.columns`: the column names. `df.shape`: (rows, columns).
- `df["J-D"]`: one column (a pandas **Series**, like a labeled 1D array). `df["J-D"].to_numpy()` gives a plain NumPy array.
- `df[df["Year"] >= 1980]`: rows selected with a boolean mask, exactly like NumPy.
- `df.describe()`: summary statistics of every numeric column.
- `df.dropna(subset=["J-D"])`: drop rows where that column is missing.

In [ ]:
print(df.shape)
print(list(df.columns))
df[["Year", "Jan", "Jul", "J-D"]].describe()

**Grouping** is a pandas superpower: split the rows into groups, compute something per group, and combine. Here we add a `Decade` column (`//` is integer division) and average the annual anomaly by decade.

In [ ]:
df["Decade"] = (df["Year"] // 10) * 10
df.groupby("Decade")["J-D"].mean().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df["Year"], df["J-D"], "o-", ms=3, label="annual mean")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("year")
ax.set_ylabel("temperature anomaly (°C)")
ax.set_title("GISTEMP global mean (relative to 1951–1980)")
ax.legend()
plt.show()

### Fitting the recent trend

Since about 1980 the rise looks roughly linear. Fit a straight line to 1980 onward (dropping the incomplete final year). Measure years from 2000 so the intercept is the anomaly in 2000 and is nearly uncorrelated with the slope.

The file contains **no uncertainties**, so we estimate a common $\sigma$ from the scatter (no `sigma`, default `absolute_sigma=False`). That makes the reduced $\chi^2$ meaningless here, as discussed. A further caveat: neighboring years are not independent (e.g. El Niño lasts more than a year), so these uncertainties are somewhat optimistic.

In [ ]:
recent = df[df["Year"] >= 1980].dropna(subset=["J-D"])
yr = recent["Year"].to_numpy()
anom = recent["J-D"].to_numpy()

p_T, cov_T = curve_fit(line, yr - 2000, anom)       # no sigma: errors estimated from scatter
err_T = np.sqrt(np.diag(cov_T))
print(f"anomaly in 2000: {p_T[0]:.3f} +/- {err_T[0]:.3f} °C")
print(f"trend: {10 * p_T[1]:.3f} +/- {10 * err_T[1]:.3f} °C per decade  ({len(yr)} years)")
print("correlation:", round(cov_T[0, 1] / (err_T[0] * err_T[1]), 3))
sigma_hat = np.std(anom - line(yr - 2000, *p_T), ddof=2)
print(f"scatter about the line: {sigma_hat:.3f} °C")

(`np.std(..., ddof=2)` divides by $N-2$ instead of $N$, matching the $\hat\sigma$ formula above with two fitted parameters.)

### Your turn 2

1. Fit a straight line to the years 1880–1940 and compare the trend (°C per decade) with the post-1980 trend.
2. Repeat the post-1980 fit using only the June–August column `JJA`. Is the summer trend consistent with the annual one within uncertainties?
3. Make a plot of the data with both fitted lines drawn over the years they cover.

In [ ]:
# Your code here

## 10. Real data: Millikan's measurement of Planck's constant

*Physics recap.* In the photoelectric effect, light of frequency $\nu$ knocks electrons out of a metal. Einstein's equation says the maximum kinetic energy of the electrons is $h\nu - \phi$, where $\phi$ is the metal's work function. The stopping voltage $V$ needed to halt the fastest electrons satisfies $eV = h\nu - \phi$, so

$$V = \frac{h}{e}\,\nu - \frac{\phi}{e} .$$

A plot of $V$ vs $\nu$ is a straight line with slope $h/e$. Millikan's 1916 measurements of exactly this gave one of the first precise values of $h$. The file has two columns: frequency (Hz) and stopping voltage (V).

In [ ]:
nu, V = np.loadtxt(data_path("millikan.txt"), unpack=True)
for f_i, V_i in zip(nu, V):
    print(f"{f_i:.4e} Hz   {V_i:.4f} V")

The file gives **no uncertainties**. Without error bars we cannot test the straight-line model with $\chi^2$; we have to *assume* it is right and estimate the measurement scatter from the residuals. With only 6 points and 2 parameters, that estimate is itself rough (only 4 degrees of freedom), so treat the final uncertainty as approximate.

`np.polyfit(..., cov=True)` does exactly this scaling.

In [ ]:
(slope, intercept), cov_m = np.polyfit(nu, V, 1, cov=True)
err_slope, err_int = np.sqrt(np.diag(cov_m))

h_fit = const.e * slope
h_err = const.e * err_slope
print(f"slope h/e = {slope:.4e} +/- {err_slope:.1e} V s")
print(f"h = {h_fit:.3e} +/- {h_err:.1e} J s")
print(f"accepted h = {const.h:.4e} J s   -> difference = {(h_fit - const.h)/h_err:.1f} sigma")
print(f"work function phi = {-intercept:.2f} +/- {err_int:.2f} eV")

resid = V - np.polyval([slope, intercept], nu)
print(f"estimated scatter of a single V measurement: {np.std(resid, ddof=2):.4f} V")

`np.polyval(coefficients, x)` evaluates the polynomial returned by `polyfit`. Note how the work function in eV is just minus the intercept in volts, since $\phi/e$ in volts equals $\phi$ in electron-volts.

The result agrees with the modern (exact, since 2019) value of $h$ to about 1%, within its uncertainty. Not bad for 1916!

In [ ]:
sig_V = np.full(V.size, np.std(resid, ddof=2))      # estimated error bar for plotting
plot_fit_with_pulls(nu / 1e14, V, sig_V, lambda x, a, b: a * x + b,
                    [slope * 1e14, intercept], r"frequency $\nu$ ($10^{14}$ Hz)", "stopping voltage $V$ (V)");

Here `lambda x, a, b: a * x + b` defines a small unnamed function on the spot (a **lambda function**), handy when a function is needed only once. The trailing `;` suppresses the printed return value.

### Your turn 3

1. Fit the Millikan data with `curve_fit` and no `sigma`. Do you get the same $h$ and uncertainty as `polyfit`?
2. What is the correlation coefficient between slope and intercept? Why is it so close to $-1$? (Hint: where is $\nu = 0$ relative to the data?)
3. Suppose Millikan had quoted an uncertainty of 0.01 V on each voltage. Refit with that `sigma` and `absolute_sigma=True`, and compute $\chi^2/\nu$ and the $p$-value. Would that claimed uncertainty be believable?

In [ ]:
# Your code here

## 11. Real data: an exponential decay

The file `decay_data.txt` contains a measured decay curve: time, measured activity, and the uncertainty of each activity value (units are not specified, so we call them "arbitrary units"). The model you would first try is pure exponential decay,

$$A(t) = A_0\, e^{-t/\tau} .$$

A semilog plot (log $y$ axis) turns an exponential into a straight line, which makes deviations easy to see.

In [ ]:
t_d, A_d, sig_d = np.loadtxt(data_path("decay_data.txt"), unpack=True)
print(f"{t_d.size} points, t from {t_d.min()} to {t_d.max()}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(t_d, A_d, yerr=sig_d, fmt=".", capsize=2)
ax.set_yscale("log")
ax.set_xlabel("time (arb. units)")
ax.set_ylabel("activity (arb. units)")
plt.show()

On the log scale, the data are straight at first but level off at late times instead of continuing downward. Let's see what the fit says. Initial guesses: $A_0 \approx$ the first point; $\tau$ is roughly where the activity has dropped to $1/e$ of its start, which by eye is around 4.

In [ ]:
def expo(t, A0, tau):
    return A0 * np.exp(-t / tau)

p_e, cov_e = curve_fit(expo, t_d, A_d, p0=[A_d[0], 4], sigma=sig_d, absolute_sigma=True)
print("A0, tau =", p_e.round(3), "+/-", np.sqrt(np.diag(cov_e)).round(3))
fit_quality(expo, t_d, A_d, sig_d, p_e)
plot_fit_with_pulls(t_d, A_d, sig_d, expo, p_e, "time (arb. units)", "activity (arb. units)");

$\chi^2/\nu \approx 7$ and the pulls are mostly positive at late times: the pure exponential is the wrong model. Real detectors always see some **background** (cosmic rays, natural radioactivity), which shows up as a constant floor. Add a constant $B$:

$$A(t) = A_0 e^{-t/\tau} + B .$$

In [ ]:
def expo_bkg(t, A0, tau, B):
    return A0 * np.exp(-t / tau) + B

p_b, cov_b = curve_fit(expo_bkg, t_d, A_d, p0=[A_d[0], 4, 0.1], sigma=sig_d, absolute_sigma=True)
err_b = np.sqrt(np.diag(cov_b))
for name, val, err in zip(["A0", "tau", "B"], p_b, err_b):
    print(f"{name:>4} = {val:.4f} +/- {err:.4f}")
fit_quality(expo_bkg, t_d, A_d, sig_d, p_b)
plot_fit_with_pulls(t_d, A_d, sig_d, expo_bkg, p_b, "time (arb. units)", "activity (arb. units)");

Now $\chi^2/\nu \approx 1$, the $p$-value is reasonable, and the pulls look random. Notice that adding the background also changed $\tau$ a lot: the pure-exponential fit had stretched $\tau$ to try to follow the flat tail. **A wrong model gives wrong parameters, not just a bad $\chi^2$.**

The **decay constant** is $\lambda = 1/\tau$. Its uncertainty follows from **error propagation**: for a function $g(\tau)$, a small error $\delta\tau$ produces $\delta g \approx |dg/d\tau|\,\delta\tau$. With $g = 1/\tau$, $|dg/d\tau| = 1/\tau^2$, so $\sigma_\lambda = \sigma_\tau / \tau^2$ (equivalently, the *relative* errors are equal: $\sigma_\lambda/\lambda = \sigma_\tau/\tau$).

In [ ]:
tau_fit, tau_err = p_b[1], err_b[1]
lam = 1 / tau_fit
lam_err = tau_err / tau_fit**2
print(f"tau    = {tau_fit:.3f} +/- {tau_err:.3f}  (arb. time units)")
print(f"lambda = {lam:.4f} +/- {lam_err:.4f}  (per arb. time unit)")
print(f"relative uncertainty: {tau_err/tau_fit:.1%}")

### Your turn 4

1. Compute the half-life $t_{1/2} = \tau\ln 2$ and its uncertainty.
2. Is the background significantly different from zero? Compute $B/\sigma_B$.
3. Look at the correlation matrix of the three parameters. Which pair is most strongly correlated, and why does that make sense physically?

In [ ]:
# Your code here

## 12. Real data: finding a planet with radial velocities

*Physics recap.* A planet and its star both orbit their common center of mass. The star moves in a small circle, so it alternately approaches and recedes from us, and the Doppler shift of its spectral lines reveals its **radial velocity** (velocity along our line of sight). For a circular orbit, the radial velocity is a sinusoid,

$$v_r(t) = K \sin\!\left(\frac{2\pi t}{P} + \varphi\right) + v_0,$$

with semi-amplitude $K$, orbital period $P$, phase $\varphi$, and an offset $v_0$.

The file `radial_motion_star.tbl` is from the NASA Exoplanet Archive. **Always look at a file before loading it.** The cell below reads the whole file as one string (from GitHub with `urllib.request` if needed, otherwise with `open`), splits it into lines, and prints the first few. The `with` statement closes the file automatically when the block ends.

In [ ]:
import urllib.request

path = data_path("radial_motion_star.tbl")
if path.startswith("http"):                       # running on Colab: read from GitHub
    text = urllib.request.urlopen(path).read().decode()
else:                                             # local copy of the file
    with open(path) as fh:
        text = fh.read()
lines = text.splitlines()                         # list of lines (strings)
print(len(lines), "lines in the file. The first 7:")
for line_text in lines[:7]:
    print(line_text)

The header lines begin with `#` and use `|` as column separators; data lines are whitespace-separated numbers. The columns are:

- **BJD**: Barycentric Julian Date, the time in days (a continuous day count, corrected to the solar system's center of mass);
- **Radial_Velocity** and **Radial_Velocity_Uncertainty** in m/s;
- **Accepted**: 1 if the measurement passed quality cuts, 0 if not.

`np.genfromtxt(..., comments="#")` skips everything after a `#`, which removes the header. Keep only accepted measurements, and subtract the first time to get manageable numbers.

In [ ]:
rv = np.genfromtxt(data_path("radial_motion_star.tbl"), comments="#")
print("shape:", rv.shape, " accepted values:", np.unique(rv[:, 3]))
rv = rv[rv[:, 3] == 1]
t_rv = rv[:, 0] - rv[0, 0]        # days since first observation
v_rv, s_rv = rv[:, 1], rv[:, 2]
print(f"{t_rv.size} measurements over {t_rv.max():.0f} days; typical error {np.median(s_rv):.1f} m/s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].errorbar(t_rv, v_rv, yerr=s_rv, fmt=".", ms=3)
axes[0].set_title("all data")
first = t_rv < 15
axes[1].errorbar(t_rv[first], v_rv[first], yerr=s_rv[first], fmt="o", ms=3)
axes[1].set_title("first 15 days")
for ax in axes:
    ax.set_xlabel("time (days)")
    ax.set_ylabel("radial velocity (m/s)")
plt.show()

The full data set spans years with big gaps (the star is only visible part of the year), so the oscillation is hidden. Zooming in on the first 15 days, there is clearly a fast oscillation with an amplitude of roughly 50 m/s and a period of a few days.

### Finding the period with a periodogram

A sinusoid fit is very sensitive to the starting period. With data spread over 2000 days, a period guess that is off by even 0.01 days puts the model completely out of phase by the end, and `curve_fit` will get lost. We need a robust first estimate.

A **periodogram** tries every period on a fine grid and measures how well a sinusoid of that period matches the data; the best period gives a tall peak. SciPy's `scipy.signal.lombscargle(t, y, omega)` computes the Lomb–Scargle periodogram, which works for unevenly spaced times. It takes **angular** frequencies $\omega = 2\pi/P$. The peaks are very narrow (width roughly $P^2/T_\text{total}$), so the grid must be fine: we step in frequency $f = 1/P$ by a tenth of $1/T_\text{total}$.

In [ ]:
from scipy.signal import lombscargle

T_total = t_rv.max()
freq = np.arange(1 / 20, 1 / 1.1, 0.1 / T_total)    # cycles per day: periods 1.1 to 20 days
power = lombscargle(t_rv, v_rv - v_rv.mean(), 2 * np.pi * freq)
P_best = 1 / freq[np.argmax(power)]
print(f"{freq.size} trial frequencies; best period = {P_best:.4f} days")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(1 / freq, power)
ax.set_xscale("log")
ax.set_xlabel("trial period (days)")
ax.set_ylabel("periodogram power")
plt.show()

One dominant peak at about 4.23 days. Now fit with `curve_fit`, using the periodogram period, the eyeballed amplitude, and the mean velocity as starting guesses. A rough phase guess is fine once the period is right.

In [ ]:
def rv_model(t, K, P, phi, v0):
    return K * np.sin(2 * np.pi * t / P + phi) + v0

p0_rv = [50, P_best, 0, v_rv.mean()]
p_rv, cov_rv = curve_fit(rv_model, t_rv, v_rv, p0=p0_rv, sigma=s_rv, absolute_sigma=True)
err_rv = np.sqrt(np.diag(cov_rv))
for name, val, err in zip(["K (m/s)", "P (days)", "phi", "v0 (m/s)"], p_rv, err_rv):
    print(f"{name:>9} = {val:.5f} +/- {err:.5f}")
chisq_rv, ndf_rv = fit_quality(rv_model, t_rv, v_rv, s_rv, p_rv)

The period is measured to a few seconds because the data span hundreds of orbits. (If $K$ comes out negative, that just means the phase shifted by $\pi$; the fit is equally good.)

### Checking the fit: phase folding

A plot of all 2000 days is useless, but since the signal repeats every $P$, we can **fold** the data: compute each observation's **phase**, its position within the cycle, $\text{phase} = (t \bmod P)/P \in [0, 1)$. If the period is right, all orbits stack onto a single clean sinusoid. `%` is the modulo (remainder) operator and works on floats.

In [ ]:
K_fit, P_fit, phi_fit, v0_fit = p_rv
phase = (t_rv % P_fit) / P_fit

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax1.errorbar(phase, v_rv, yerr=s_rv, fmt=".", ms=3, alpha=0.7, label="data, folded")
ph = np.linspace(0, 1, 300)
ax1.plot(ph, rv_model(ph * P_fit, *p_rv), "C1-", lw=2, label="fit")
ax1.set_ylabel("radial velocity (m/s)")
ax1.legend()
ax2.plot(phase, (v_rv - rv_model(t_rv, *p_rv)) / s_rv, ".", ms=3)
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xlabel(f"orbital phase (P = {P_fit:.4f} d)")
ax2.set_ylabel("pull")
plt.show()

The folded data trace one sinusoid. Note $\chi^2/\nu \approx 1.3$ with a tiny $p$-value. With 256 points, even a modest excess is very significant. The pulls show no pattern, so this is most likely extra random scatter not included in the error bars: stars' surfaces boil and pulsate, adding a few m/s of "stellar jitter." In practice astronomers add a jitter term in quadrature to each $\sigma_i$.

### How heavy is the planet?

For a circular orbit and a planet much lighter than its star, momentum balance plus Kepler's third law give the planet's mass:

$$m_p \sin i = K \left(\frac{P}{2\pi G}\right)^{1/3} M_\star^{2/3},$$

where $M_\star$ is the star's mass and $i$ is the orbit's inclination to our line of sight, which radial velocities alone cannot determine. That is why this is a **minimum mass** ($\sin i \le 1$).

The data file does not identify the star, so we must **assume** its mass. Let's take a Sun-like star, $M_\star = 1\,M_\odot$, and note that the answer scales as $M_\star^{2/3}$ (a 10% error in $M_\star$ gives a 7% error in $m_p$). For the Sun we use the IAU nominal value $GM_\odot = 1.3271244\times10^{20}\ \text{m}^3/\text{s}^2$, and the Jupiter-to-Sun mass ratio $1/1047.6$.

In [ ]:
GM_sun = 1.3271244e20          # m^3/s^2 (IAU 2015 nominal)
M_sun_over_M_jup = 1047.6

K_si = abs(K_fit)              # m/s
P_si = P_fit * 86400           # s
M_star = 1.0                   # ASSUMED, in solar masses

# m_p sin i in solar masses: K (P / (2 pi G M_sun))^(1/3) * M_star^(2/3)
m_sini_sun = K_si * (P_si / (2 * np.pi * GM_sun))**(1 / 3) * M_star**(2 / 3)
print(f"m_p sin i = {m_sini_sun * M_sun_over_M_jup:.3f} Jupiter masses (for M_star = {M_star} M_sun)")

a_orbit = (GM_sun * M_star * P_si**2 / (4 * np.pi**2))**(1 / 3)    # Kepler's third law
print(f"orbital radius = {a_orbit / const.au:.4f} AU")

About half a Jupiter mass, orbiting at 0.05 AU (Mercury is at 0.39 AU) with a 4-day "year": a **hot Jupiter**. The discovery of exactly this kind of planet around a Sun-like star in 1995 earned the 2019 Nobel Prize in Physics.

### Your turn 5

1. Try the sinusoid fit with a starting period of 4.0 days instead of the periodogram value. What happens? Plot the phase-folded data with the resulting period.
2. Add a "jitter" $s_j$ in quadrature, $\sigma_i \to \sqrt{\sigma_i^2 + s_j^2}$, and find (by trying a few values) the $s_j$ that makes $\chi^2/\nu \approx 1$. How do the uncertainties on $K$ and $P$ change?

In [ ]:
# Your code here

## 13. Fitting a histogram

Often the data are not $(x, y \pm \sigma)$ points but a list of individual events, for example the reconstructed mass of each particle candidate in a collider experiment. To fit, you histogram the events and fit the bin counts.

*What is the error on a bin count?* If events land in a bin at random with some average rate, the count $n$ follows the **Poisson distribution** from last week, whose standard deviation is $\sqrt{\mu} \approx \sqrt{n}$. So the usual choice is $\sigma_i = \sqrt{n_i}$.

### A simulated resonance

*Physics recap.* An unstable particle (a **resonance**) does not have a perfectly sharp mass. Because it lives only a short time $\tau$, the energy–time uncertainty relation gives its mass a spread of width $\Gamma = \hbar/\tau$. The measured mass follows the **Breit–Wigner** (Cauchy) distribution

$$p(m) = \frac{1}{\pi}\,\frac{\Gamma/2}{(m - M)^2 + (\Gamma/2)^2},$$

peaked at $M$ with full width at half maximum $\Gamma$.

The following is a **simulated** data set, loosely inspired by the $Z$ boson ($M = 91.19$ GeV, $\Gamma = 2.50$ GeV) decaying to two muons: 2000 signal events plus 1000 background events falling exponentially with mass. The Breit–Wigner can be sampled with the inverse-transform method: its CDF inverts to $m = M + \tfrac{\Gamma}{2}\tan[\pi(u - \tfrac12)]$. (Real detectors also smear the mass with a finite resolution; we ignore that here.)

In [ ]:
M_true, G_true = 91.19, 2.50          # GeV
lo, hi = 70.0, 110.0                  # mass window, GeV

def simulate_events(n_sig, n_bkg, rng):
    """Simulated invariant masses: Breit-Wigner signal + exponential background in [lo, hi]."""
    sig = M_true + G_true / 2 * np.tan(np.pi * (rng.random(3 * n_sig) - 0.5))
    sig = sig[(sig > lo) & (sig < hi)][:n_sig]                 # keep those inside the window
    slope = 15.0                                               # GeV, background fall-off
    u = rng.random(n_bkg)
    bkg = lo - slope * np.log(1 - u * (1 - np.exp(-(hi - lo) / slope)))   # truncated exponential
    return np.concatenate([sig, bkg])

masses = simulate_events(2000, 1000, rng)
print(masses.size, "simulated events")

The model for the expected count in a bin centered at $m$ with width $w$ is the number of events times the PDF times the bin width:

$$\mu(m) = N_s\, w\, \frac{1}{\pi}\frac{\Gamma/2}{(m-M)^2 + (\Gamma/2)^2} + B\, e^{-(m - 70\,\text{GeV})/s}.$$

Here $N_s$ is the signal normalization (the Breit–Wigner tails extend beyond the window, so $N_s$ is slightly larger than the number of signal events inside it), and $B$, $s$ describe the background.

In [ ]:
edges = np.linspace(lo, hi, 81)                   # 80 bins of 0.5 GeV
centers = 0.5 * (edges[:-1] + edges[1:])
width = edges[1] - edges[0]
counts, _ = np.histogram(masses, bins=edges)

def peak_model(m, Ns, M, G, B, s):
    bw = (G / 2) / np.pi / ((m - M)**2 + (G / 2)**2)
    return Ns * width * bw + B * np.exp(-(m - lo) / s)

print("empty bins:", np.sum(counts == 0))
sig_counts = np.sqrt(counts)
p0_pk = [2000, centers[np.argmax(counts)], 3, counts[0], 15]
p_pk, cov_pk = curve_fit(peak_model, centers, counts, p0=p0_pk, sigma=sig_counts, absolute_sigma=True)
err_pk = np.sqrt(np.diag(cov_pk))
for name, val, err in zip(["Ns", "M (GeV)", "Gamma (GeV)", "B", "s (GeV)"], p_pk, err_pk):
    print(f"{name:>12} = {val:9.3f} +/- {err:.3f}")
fit_quality(peak_model, centers, counts, sig_counts, p_pk);

In [ ]:
plot_fit_with_pulls(centers, counts, sig_counts, peak_model, p_pk,
                    r"dimuon mass $m$ (GeV)", "events per 0.5 GeV");

The fit recovers $M$ and $\Gamma$ within their uncertainties. With thousands of events, the $\sqrt{n}$ errors work well.

### Trouble at low statistics

Now take a small sample, 60 signal and 30 background events, with the same binning. Many bins contain 0 or 1 events.

In [ ]:
small = simulate_events(60, 30, rng)
counts_s, _ = np.histogram(small, bins=edges)
print("bin counts:", counts_s)
print("empty bins:", np.sum(counts_s == 0), "of", counts_s.size)

With $\sigma_i = \sqrt{n_i}$, every empty bin has $\sigma_i = 0$, and $\chi^2$ divides by zero. Even with the *true* parameters, $\chi^2$ is infinite. (`np.errstate(divide="ignore")` just silences NumPy's divide-by-zero warning inside the `with` block.)

In [ ]:
with np.errstate(divide="ignore"):
    chi2_true = np.sum((counts_s - peak_model(centers, 90, M_true, G_true, 1, 15))**2 / counts_s)
print("chi2 at the true parameters:", chi2_true)

If you pass such `sigma` values to `curve_fit`, it prints divide-by-zero warnings and returns meaningless numbers, so you must do something about the empty bins.

Common hacks are to drop the empty bins, or to set their error to 1. Both are biased:

- Dropping empty bins throws away the information "nothing was seen here," so nothing stops the fitted curve from floating up in sparse regions.
- More fundamentally, $\sqrt{n_i}$ is a bad estimate of the true error when $n_i$ is small. A bin that fluctuated *down* gets a small error bar and therefore a large weight, while a bin that fluctuated *up* gets a small weight. That systematically pulls the fit low.
- The Poisson distribution for small means is not Gaussian at all (it is lopsided and cannot go below zero), but least squares implicitly assumes Gaussian errors.

You can see the effect directly by fitting with the empty bins dropped and comparing the fitted number of events with the true total.

In [ ]:
keep = counts_s > 0
p_s, cov_s = curve_fit(peak_model, centers[keep], counts_s[keep], p0=[60, 91, 3, 1, 15],
                       sigma=np.sqrt(counts_s[keep]), absolute_sigma=True, maxfev=20000)
total_fit = np.sum(peak_model(centers, *p_s))
print(f"events in histogram: {counts_s.sum()};  fitted model predicts {total_fit:.1f} in the window")
print(f"fitted M = {p_s[1]:.2f} GeV, Gamma = {p_s[2]:.2f} GeV")

The fitted curve predicts noticeably more events than were actually observed, even though it was supposed to describe this very histogram: here the dropped empty bins win over the downward pull. Either way, the answer is biased, and the fitted width is unreliable too. Next week you will fix this properly with the **maximum likelihood** method, which uses the exact Poisson probabilities and works even with empty bins, or with no bins at all.

### Your turn 6

Go back to the large simulated sample (`masses`).

1. Refit using 20 bins and then 200 bins over the same range. How do the fitted $M$ and $\Gamma$ and their uncertainties change? How many empty bins appear with 200 bins?
2. With 80 bins, how much does the fitted $\Gamma$ change if you ignore the background (fit only the Breit–Wigner term)? Look at the pulls.

In [ ]:
# Your code here

## Summary

- An error bar $\sigma$ is the standard deviation of a measurement: about 68% of points should lie within $1\sigma$ of the true curve.
- **Least squares:** minimize $\chi^2 = \sum_i [(y_i - f(x_i))/\sigma_i]^2$. For a straight line the solution is analytic (normal equations).
- `np.polyfit(x, y, deg, w=1/sigma, cov="unscaled")`: polynomial fits; coefficients highest power first; weights are $1/\sigma$, not $1/\sigma^2$.
- `popt, pcov = curve_fit(f, x, y, p0=..., sigma=..., absolute_sigma=True)`: any model. Uncertainties are `np.sqrt(np.diag(pcov))`.
- The **covariance matrix**: variances on the diagonal, covariances off it; correlation $\rho_{ij} = C_{ij}/(\sigma_i\sigma_j)$. Centering $x$ reduces correlations.
- **Goodness of fit:** $\nu = N - n_\text{par}$; expect $\chi^2/\nu \approx 1$; $p$-value from `scipy.stats.chi2.sf(chisq, ndf)`.
- Always plot the fit **with pulls**. Patterns mean a wrong model; pulls too large or small mean wrong error bars.
- Without error bars, estimate $\sigma$ from the residual scatter (`polyfit(..., cov=True)`, `curve_fit` without `sigma`); then $\chi^2$ cannot test the model.
- Nonlinear fits need good **initial guesses**: read them off the data, use a periodogram for periods, and always plot the result.
- **pandas:** `pd.read_csv(..., skiprows=, na_values=)`, `df["col"]`, `df[mask]`, `df.describe()`, `df.dropna()`, `df.groupby("col")["other"].mean()`.
- **Histogram fits:** model = expected counts per bin; $\sigma_i = \sqrt{n_i}$ fails for small counts and empty bins. That motivates maximum likelihood.
- Results today: $h = (6.55 \pm 0.08)\times10^{-34}$ J s from Millikan's data; a hot Jupiter with $P \approx 4.23$ days.

## Further reading

- SciPy `curve_fit` documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
- J. VanderPlas, *Python Data Science Handbook*, Chapter 3 (pandas), free at https://jakevdp.github.io/PythonDataScienceHandbook/
- J. R. Taylor, *An Introduction to Error Analysis*, chapters 8 and 12 (least squares and $\chi^2$) — a classic undergraduate text.
- NASA Exoplanet Archive: https://exoplanetarchive.ipac.caltech.edu ; NASA GISTEMP: https://data.giss.nasa.gov/gistemp/